In [4]:
import re
import pandas as pd 

from pathlib import Path


pd.set_option("display.max_colwidth", None)

RAW_FILE = Path("../data/raw/pao_metadata/Book1.xlsx")
REVIEW_FILE = Path("../data/interim/pao_label_review.csv")
OUTPUT_FILE = Path("../data/processed/pao_metadata_v1.parquet")

print(f"raw file", RAW_FILE.exists(),  "\nreview file: ",  REVIEW_FILE.exists(), "\nOutput File: ", OUTPUT_FILE.exists())

raw file True 
review file:  False 
Output File:  False


In [5]:
HEADER_ROWS = {
    "COMMUNICATION": 2,
    "CLEARANCE": 0,
    "AOsMEMOs": 2,
    "LEAVES": 0,
}

sheets={}


for name, header in HEADER_ROWS.items():
    df = pd.read_excel(RAW_FILE, sheet_name=name, header=header)
    df.columns = df.columns.astype(str).str.strip()
    sheets[name] = df
    print(f"{name:15} rows={df.shape[0]:5} columns={df.shape[1]}")


COMMUNICATION   rows= 2285 columns=12
CLEARANCE       rows=  702 columns=16
AOsMEMOs        rows=   14 columns=6
LEAVES          rows=  657 columns=9


In [6]:
for name, df in sheets.items():
    print(f"\n---{name}---")
    print(df.columns.tolist())

for name, df in sheets.items():
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]

    if not unnamed:
        print(f"{name:15} no Unnamed Columns")
        continue

    print(f"\n{name}:")
    for col in unnamed:
        filled=df[col].notna().sum()
        print(f" {col:12} values: {filled}")
        if filled > 0:
            display(df.loc[df[col].notna(), [col]].head(5))


comm = sheets["COMMUNICATION"]
rows = comm["Unnamed: 10"].notna()


display(comm.loc[rows, ["SUBJECT","DESCRIPTION", "Unnamed: 10"]])

subject = comm["SUBJECT"].fillna("").astype(str).str.strip().str.upper()
desc    = comm["DESCRIPTION"].fillna("").astype(str).str.strip().str.upper()

same       = subject == desc
desc_empty = desc == ""

# subject_in_desc = [s in d for s, d in zip(subject, desc)]
subject_in_desc = [s != "" and s in d for s, d in zip(subject, desc)]

print("rows:                        ", len(comm))
print("DESCRIPTION same as SUBJECT: ", same.sum())
print("DESCRIPTION empty:           ", desc_empty.sum())
print("SUBJECT contained in DESC:   ", sum(subject_in_desc))


---COMMUNICATION---
['DATE', 'CONTROL#', 'SUBJECT', 'DESCRIPTION', 'TYPE', 'OFFICE/PERSON FROM', 'OFFICE/UNIT CONCERNED', 'REMARKS', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

---CLEARANCE---
['DATE', 'CONTROL#', 'FULL N AME', 'OFFICE', 'PURPOSE', 'REMARKS', 'RELEASING DATE', 'C/O', 'REMARKS 2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15']

---AOsMEMOs---
['DATE RECEIVED', 'DOCUMENT DATE', 'AO/MEMO/EO #', 'SUBJECT', 'OFFICE/UNIT/PERSON FROM', 'REMARKS']

---LEAVES---
['DATE FILED', 'TRANSMITAL DATE', 'CONTROL#', 'NAME', 'TYPE OF LEAVE', 'REMARKS', 'DAY/S APPLIED', 'DATE RECEIVED', 'STATUS']

COMMUNICATION:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 3


,Unnamed: 10
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


 Unnamed: 11  values: 3


,Unnamed: 11
446,"THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN)



CLEARANCE:
 Unnamed: 9   values: 0
 Unnamed: 10  values: 0
 Unnamed: 11  values: 0
 Unnamed: 12  values: 0
 Unnamed: 13  values: 0
 Unnamed: 14  values: 0
 Unnamed: 15  values: 1


,Unnamed: 15
272,


AOsMEMOs        no Unnamed Columns
LEAVES          no Unnamed Columns


,SUBJECT,DESCRIPTION,Unnamed: 10
446,"THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE AMOUNT THREE HUNDRED FORTY NINE MILLION SIX HUNDRED FIFTY THREE THOUSAND TWO HUNDRED SEVENTEEN PESOS ONLY (349,653,217.00) HAS BEEN CREDITED TO YOUR ACCOUNT ON JANUARY 31, 2025.","THE CONFIRMATION LETTER FROM THE AUDIT TEAM LEADER OF DILG-OSEC AUDIT TEAM TO VERIFY THE FUND TRANSFER TO THE PROVINCIAL GOVERNMENT OF CAGAYAN AMOUNTING TO P114,846.20. FOR YOUR REFERENCE AND COMPLIANCE"
448,"INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","INVITATION - 2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY.","2024 PROVINCIAL PROGRAM IMPLEMENTATION REVIEW AND 2025 PLANNING WORKSHOP ON FEBRUARY 11, 2025, 8:30 AM- 5:00 PM, AT G-VIEW EVENTS PLACE, CAGGAY, TUGUEGARAO CITY."
481,ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),ORIENTATION ON KONSULTA PACKAGE: AN INITIATIVE TOWARDS A FUNCTIONAL HEALTH CARE PROVIDER NETWORK (HCPN),Orientation on Konsulta Package: An Initiative Towards a Functional Health Care Provider Network (HCPN)


rows:                         2285
DESCRIPTION same as SUBJECT:  1604
DESCRIPTION empty:            0
SUBJECT contained in DESC:    1722


In [7]:
#remove empty columns and rows
for name, df in sheets.items():
    before = df.shape

    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    sheets[name]=df
    print(f"{name:15} before={before} after={df.shape}")


COMMUNICATION   before=(2285, 12) after=(2285, 9)
CLEARANCE       before=(702, 16) after=(702, 9)
AOsMEMOs        before=(14, 6) after=(14, 6)
LEAVES          before=(657, 9) after=(653, 9)


In [8]:
#give every row a permanent ID
for name, df in sheets.items():
    df=df.reset_index(drop=True)
    df["row_id"] = name + "-" + df.index.astype(str)
    sheets[name]= df

display(sheets["COMMUNICATION"][["row_id", "SUBJECT"]].head(3))


#adds IDs like COMMUNICATION-0 that stay sttached to the same record so saved review decisions always match


,row_id,SUBJECT
0,COMMUNICATION-0,REQUEST TO RENDER OVERTIME
1,COMMUNICATION-1,PFC MEETING
2,COMMUNICATION-2,NOTICE OF ASSIGNMENT/ GRANT OF SECURITY INTEREST


In [9]:
#choose the input columns
TEXT_COLUMNS = {
    "COMMUNICATION": ["SUBJECT", "DESCRIPTION"],
    "CLEARANCE": ["PURPOSE"],
    "AOsMEMOs": ["SUBJECT"],
    "LEAVES": ["TYPE OF LEAVE", "REMARKS"],
}

for name, cols in TEXT_COLUMNS.items():
    missing=[c for c in cols if c not in sheets[name].columns]
    print(f"{name:15} using {cols} missing: {missing}")

    

COMMUNICATION   using ['SUBJECT', 'DESCRIPTION'] missing: []
CLEARANCE       using ['PURPOSE'] missing: []
AOsMEMOs        using ['SUBJECT'] missing: []
LEAVES          using ['TYPE OF LEAVE', 'REMARKS'] missing: []


In [10]:
#build document_text without repeated text
def join_unique(values):
    result = ""
    for value in values:
        value =str(value).strip()
        if value == "" or value.lower() == "nan":
            continue
        if value.upper() in result.upper():
            continue
        if result.upper() in value.upper():
            result=value
        else:
            result=(result + " " + value).strip()
    return result

parts = []
for name, cols in TEXT_COLUMNS.items():
    df = sheets[name]
    text = df[cols].fillna("").apply(join_unique, axis=1)
    parts.append(pd.DataFrame({
        "row_id": df["row_id"],
        "document_text": text,
        "label":name,
    }))

datasets=pd.concat(parts, ignore_index=True)

print(datasets.shape)
print(datasets["label"].value_counts())
display(datasets.sample(5, random_state=42))


#this joins each row's chosen columns into one text, skipping repeats, then stacks the 4 sheets into one table called datasets
#check: about 3650 rows; COMMUNICATION ~ 2,285,         

(3654, 3)
label
COMMUNICATION    2285
CLEARANCE         702
LEAVES            653
AOsMEMOs           14
Name: count, dtype: int64


,row_id,document_text,label
1313,COMMUNICATION-1313,"MEMORANDUM ORDER NO. 2025-11-11-001 APPROVED SCHEDULE OF DUTY ASSIGNMENT @ BANGAG, LALLO",COMMUNICATION
1674,COMMUNICATION-1674,REQUEST FOR CHANGE OF PAYEE NAME REQUEST THAT THE ISSUED CHECKS AND FUTURE PAYMENT BE MADE PAYABLE TO NEW TUGUEGARAO BOMBAY BAZAAR CORPORATIOM OR LACHMIII RAICHANDANI OR KUMAR ADTANI,COMMUNICATION
229,COMMUNICATION-229,"AUDIT OBSERVATIONS AND RECOMMENDATIONS(AOM) NO.2024-009(2023) DATED APRIL 1,2024",COMMUNICATION
3509,LEAVES-508,WELLNESS LEAVE,LEAVES
2222,COMMUNICATION-2222,"TRAINING ON DATA ANALYSIS AND VISUALIZATION SOFTWARE ENTITLED""INTERACTIVE DASHBOARD: ""TRANSFORMING DATA INTO DYNAMIC INSIGHTS USING MS POWER BI"" BATCH 2 ON SEPTEMBER 16-18. 2026 AT GVIEW EVENT CENTER",COMMUNICATION


In [11]:
#prove the doubled sentences are gone
comm_text=datasets.loc[datasets["label"]  == "COMMUNICATION", "document_text"]


def has_repeat(text):
    half=len(text) // 2
    return len(text) > 20 and text[:half].strip() == text[half:].strip()

print("texts repeated twice:", comm_text.map(has_repeat).sum())


#checks wheter any COMMUNICATION text is the same sentence written twice
#checks : prints 0 or close to it

texts repeated twice: 0


In [13]:
#clean the text and remove blank rows
def clean_text(text):
    text = str(text).upper()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

datasets["document_text"] = datasets["document_text"].map(clean_text)


blank=datasets["document_text"].eq("")
print("blank_rows:", blank.sum())
display(datasets[blank].head())

datasets=datasets[~blank].reset_index(drop=True)
print("rows after removing blanks:", len(datasets))
print(datasets["label"].value_counts())


#what it does: clean_texts makes every text uppercase and turns runs of spaces, tabs or new lines (\s +) into one space. That way "Sick leave" and "SICK LEAVE" become the same text.
# .map(clean_text) runs the function on every row.
# blank marks the rows whose text is empty (for example, a CLEARANCE row with no PURPOSE). ~blank keeps the rows that are not blank.
# reset_index(drop=True) renumbers the rows after removing some.

blank_rows: 0


,row_id,document_text,label


rows after removing blanks: 3644
label
COMMUNICATION    2285
CLEARANCE         693
LEAVES            652
AOsMEMOs           14
Name: count, dtype: int64


In [ ]:
# dupes =  datasets["document_text"].duplicated().sum()
# print(dupes)



1219
